# 10: Production Readiness & Observability — Full Walkthrough

**Goal:** Transform the existing Ingress-based stack into a production-grade system with RBAC security, resource governance, health probes, Prometheus/Grafana monitoring via Helm, and HPA autoscaling.

**Stack:** Frontend (FastAPI + httpx) → Backend (FastAPI + psycopg2) → PostgreSQL (StatefulSet + PVC) + Ingress (NGINX) + Prometheus/Grafana (Helm) + HPA

## Architecture

```
         User (port 30080)
                │
                ▼
        ┌───────────────┐
        │ NGINX Ingress  │  path-based routing
        │ (NodePort 30080)│  /  → frontend-service:80
        └───────┬───────┘  /api → backend-service:80
                │
        ┌───────┴───────┐
        │               │
        ▼               ▼
 frontend-service   backend-service
    (ClusterIP)       (ClusterIP)
        │               │
        ▼               ▼
   Frontend Pod      Backend Pods ×2
                         │ (RBAC: ServiceAccount + Role)
                         │
                    ┌────┘
                    ▼
              postgres (headless)
                    │
                    ▼
              PostgreSQL StatefulSet
                    │
                    ▼
              PVC (1Gi)

 Monitoring Stack (Helm-installed):
 ┌─────────────────────────────────────┐
 │  Prometheus ← Grafana ← Alertmanager│
 │  (scrapes metrics)  (dashboards)    │
 └─────────────────────────────────────┘
```

**What Phase 10 adds beyond Phase 9:**
| Enhancement | Why It Matters |
|-------------|---------------|
| **Liveness/Readiness probes** | K8s knows when a container is alive vs ready to serve traffic |
| **Resource requests/limits** | Guarantees minimum resources, prevents noisy neighbors |
| **RBAC** | Backend gets a dedicated ServiceAccount with least-privilege permissions |
| **Prometheus + Grafana** | Cluster-wide metrics collection and visualization |
| **Helm** | Package manager for K8s — installs complex stacks (Prometheus stack) in one command |
| **HPA** | Auto-scales backend under CPU load |
| **Production namespace isolation** | `production` and `monitoring` namespaces keep concerns separated |

## 1. Directory Setup & Cluster

```powershell
cd C:\projects\kubernetes_lab
mkdir 10.production-readiness
cd .\10.production-readiness
mkdir k8s, backend, frontend -Force
```

### Kind Cluster with Port Mappings

`kind-config.yaml` — Maps host ports 30080 (HTTP) and 30443 (HTTPS) to the container so the Ingress controller is reachable without `kubectl port-forward`.

```yaml
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
- role: control-plane
  extraPortMappings:
  - containerPort: 30080
    hostPort: 30080
    protocol: TCP
  - containerPort: 30443
    hostPort: 30443
    protocol: TCP
```

```powershell
kind delete cluster --name k8-lab
kind create cluster --name k8-lab --config kind-config.yaml

kubectl create namespace production
kubectl create namespace monitoring
kubectl config set-context --current --namespace=production
```

> Production app goes in `production`, monitoring stack goes in `monitoring` — real-world pattern.

## 2. ConfigMap & Secret

**`k8s/config-and-secret.yaml`**

- **ConfigMap** stores non-sensitive data: `POSTGRES_DB`, `POSTGRES_USER`, `POSTGRES_HOST`, `POSTGRES_PORT`.
- **Secret** stores sensitive data: `POSTGRES_PASSWORD` (base64-encoded `admin123`).

Both are injected into Pods via `configMapKeyRef` and `secretKeyRef` — keeps credentials out of the container image.

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: backend-config
  namespace: production
data:
  POSTGRES_DB: mydb
  POSTGRES_USER: admin
  POSTGRES_HOST: postgres
  POSTGRES_PORT: "5432"
---
apiVersion: v1
kind: Secret
metadata:
  name: backend-secret
  namespace: production
type: Opaque
data:
  POSTGRES_PASSWORD: YWRtaW4xMjM=   # admin123
```

## 3. PostgreSQL StatefulSet — Probes, Resources & Persistent Storage

**`k8s/postgres.yaml`**

### Headless Service
`clusterIP: None` — each StatefulSet Pod gets a stable DNS name (`postgres-0.postgres.production.svc.cluster.local`) instead of a single VIP. Enables direct Pod-to-Pod communication.

### Resource Requests & Limits
```yaml
resources:
  requests:
    cpu: 100m
    memory: 128Mi
  limits:
    cpu: 500m
    memory: 256Mi
```
- **requests** = Kubernetes guarantees this much. Used for scheduling decisions.
- **limits** = Hard cap. CPU is throttled; memory OOM-kills the container.
- Without these, one Pod can starve others on the same node (noisy neighbor).

### Liveness & Readiness Probes
```yaml
livenessProbe:
  exec:
    command: ["pg_isready", "-U", "admin", "-d", "mydb"]
  initialDelaySeconds: 10
  periodSeconds: 5
readinessProbe:
  exec:
    command: ["pg_isready", "-U", "admin", "-d", "mydb"]
  initialDelaySeconds: 5
  periodSeconds: 5
```

| Probe | Purpose | On Failure |
|-------|---------|------------|
| **livenessProbe** | "Is the container alive?" | Kubelet restarts the container |
| **readinessProbe** | "Is it ready to serve?" | Removed from Service endpoints |

We use `exec` with `pg_isready` — checks if Postgres is actually accepting queries (more accurate than TCP socket check).

### Persistent Storage
```yaml
volumeClaimTemplates:
- metadata:
    name: postgres-storage
  spec:
    accessModes: ["ReadWriteOnce"]
    resources:
      requests:
        storage: 1Gi
```
`volumeClaimTemplates` creates a unique PVC for each replica (`postgres-storage-postgres-0`). Data survives pod restarts and rescheduling.

## 4. Backend — RBAC, Probes & Resources

**`k8s/backend.yaml`** combines 4 resources:

### ServiceAccount + Role + RoleBinding

```yaml
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: backend-sa
  namespace: production
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: backend-role
  namespace: production
rules:
- apiGroups: [""]
  resources: ["configmaps", "secrets", "pods"]
  verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: backend-rb
  namespace: production
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: backend-role
subjects:
- kind: ServiceAccount
  name: backend-sa
  namespace: production
```

**RBAC Concepts:**
- **ServiceAccount** = Identity for the backend Pod (not a human).
- **Role** = Namespace-scoped rules: read-only access to configmaps, secrets, pods.
- **RoleBinding** = Binds the Role to the ServiceAccount.

> **Principle of Least Privilege:** The backend can only read — never create, update, or delete. If compromised, blast radius is limited.

### Deployment

```yaml
spec:
  replicas: 2
  template:
    spec:
      serviceAccountName: backend-sa    # Bind RBAC identity
      containers:
      - name: api
        image: fastapi-backend:2.0
        ports:
        - containerPort: 80
        env:
        - name: POSTGRES_HOST
          valueFrom:
            configMapKeyRef:
              name: backend-config
              key: POSTGRES_HOST
        # ... (POSTGRES_DB, POSTGRES_USER, POSTGRES_PORT from ConfigMap)
        # ... (POSTGRES_PASSWORD from Secret)
        resources:
          requests:
            cpu: 50m
            memory: 64Mi
          limits:
            cpu: 200m
            memory: 128Mi
        livenessProbe:
          httpGet:
            path: /
            port: 80
          initialDelaySeconds: 5
          periodSeconds: 10
        readinessProbe:
          httpGet:
            path: /
            port: 80
          initialDelaySeconds: 3
          periodSeconds: 5
```

- **2 replicas** = High availability. If one fails, the other serves traffic.
- **HTTP probes** on `/` — FastAPI returns 200, so probes always pass when the app is healthy.
- **Env injection** from ConfigMap + Secret — no hardcoded credentials in the image.

## 5. Frontend — Probes & Resources

**`k8s/frontend.yaml`**

Same probe + resource pattern as backend:

```yaml
spec:
  replicas: 1
  template:
    spec:
      containers:
      - name: web
        image: frontend:1.0
        resources:
          requests:
            cpu: 50m
            memory: 64Mi
          limits:
            cpu: 200m
            memory: 128Mi
        livenessProbe:
          httpGet:
            path: /
            port: 80
          initialDelaySeconds: 5
          periodSeconds: 10
        readinessProbe:
          httpGet:
            path: /
            port: 80
          initialDelaySeconds: 3
          periodSeconds: 5
```

1 replica is sufficient for this demo. The frontend serves the HTML UI and proxies `/api` to the backend via httpx.

## 6. Ingress

**`k8s/ingress.yaml`** — Path-based routing through NGINX Ingress Controller:

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: main-ingress
  namespace: production
spec:
  ingressClassName: nginx
  rules:
  - http:
      paths:
      - path: /api
        pathType: Prefix
        backend:
          service:
            name: backend-service
            port:
              number: 80
      - path: /
        pathType: Prefix
        backend:
          service:
            name: frontend-service
            port:
              number: 80
```

The Ingress Controller runs in `ingress-nginx` namespace (installed separately for kind). Host port 30080 maps to container port 30080 (NodePort) which routes to the controller's port 80.

## 7. Application Code

### Backend — `backend/main.py`

Uses the modern FastAPI `lifespan` pattern (`@asynccontextmanager`) instead of the deprecated `@app.on_event("startup")`:

```python
from contextlib import asynccontextmanager
from fastapi import FastAPI
import psycopg2
import os

def get_db():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST", "postgres"),
        database=os.getenv("POSTGRES_DB", "mydb"),
        user=os.getenv("POSTGRES_USER", "admin"),
        password=os.getenv("POSTGRES_PASSWORD", "admin123"),
        port=os.getenv("POSTGRES_PORT", "5432")
    )

@asynccontextmanager
async def lifespan(app: FastAPI):
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS messages (
        id SERIAL PRIMARY KEY,
        content TEXT,
        created_at TIMESTAMP DEFAULT NOW()
    )""")
    cur.execute("SELECT COUNT(*) FROM messages")
    if cur.fetchone()[0] == 0:
        cur.execute("INSERT INTO messages (content) VALUES ('Hello from PostgreSQL!')")
    conn.commit()
    cur.close()
    conn.close()
    yield
    print("Application is shutting down.")

app = FastAPI(lifespan=lifespan)

@app.get("/")
def root():
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT content, created_at FROM messages ORDER BY id DESC LIMIT 1")
    row = cur.fetchone()
    cur.close()
    conn.close()
    if row:
        return {"message": row[0], "time": row[1].isoformat()}
    return {"message": "no data", "time": ""}
```

**Key behavior:** On startup, connects to PostgreSQL, creates the `messages` table if needed, seeds "Hello from PostgreSQL!" if empty. The `/` endpoint reads the latest message — it's also what the liveness/readiness probes hit.

### Frontend — `frontend/main.py`

```python
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import httpx

app = FastAPI()
BACKEND_URL = "http://backend-service:80"

@app.get("/", response_class=HTMLResponse)
async def index():
    return """
    <!DOCTYPE html>
    <html>
    <head><title>K8s Production App</title></head>
    <body>
        <h1>Production&#8209;grade Kubernetes App</h1>
        <div id="output">Loading...</div>
        <script>
            fetch('/api')
                .then(res => res.json())
                .then(data => {
                    document.getElementById('output').innerHTML =
                        `<p><strong>Message:</strong> ${data.message}</p>
                         <p><strong>Time:</strong> ${data.time}</p>`;
                })
                .catch(err => {
                    document.getElementById('output').innerText = 'Error: ' + err;
                });
        </script>
    </body>
    </html>
    """

@app.get("/api")
async def proxy_api():
    async with httpx.AsyncClient() as client:
        resp = await client.get(BACKEND_URL)
        return resp.json()
```

Serves HTML at `/`. JavaScript fetches `/api` (relative → goes through Ingress → backend), displays message + timestamp.

### Dockerfile (both services)

```dockerfile
FROM python:3.11-alpine
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY main.py .
EXPOSE 80
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "80"]
```

`python:3.11-alpine` (~50MB) minimizes attack surface. The separate `COPY requirements.txt` + `RUN pip install` layers leverage Docker caching — if requirements don't change, that layer is reused.

## 8. Build, Load & Deploy

### Build + Load Images into kind
```powershell
cd backend
docker build -t fastapi-backend:2.0 .
kind load docker-image fastapi-backend:2.0 --name k8-lab
cd ..

cd frontend
docker build -t frontend:1.0 .
kind load docker-image frontend:1.0 --name k8-lab
cd ..
```

> **Why `kind load docker-image`?** Kind runs nodes as Docker containers. They don't share the local Docker image cache — you must explicitly load images. Without this, you get `ErrImagePull`.

### Apply Manifests
```powershell
kubectl apply -f k8s/config-and-secret.yaml
kubectl apply -f k8s/postgres.yaml
kubectl apply -f k8s/backend.yaml
kubectl apply -f k8s/frontend.yaml

kubectl wait --for=condition=ready pod postgres-0 --timeout=60s
```

### Install Ingress Controller
```powershell
kubectl apply -f https://raw.githubusercontent.com/kubernetes/ingress-nginx/main/deploy/static/provider/kind/deploy.yaml
kubectl -n ingress-nginx wait --for=condition=ready pod --selector=app.kubernetes.io/component=controller --timeout=120s

# Set NodePort to 30080
kubectl -n ingress-nginx patch svc ingress-nginx-controller -p '{"spec":{"type":"NodePort","ports":[{"name":"http","port":80,"targetPort":80,"nodePort":30080},{"name":"https","port":443,"targetPort":443,"nodePort":30443}]}}'

kubectl apply -f k8s/ingress.yaml
```

### Test
```
http://localhost:30080/
```

Expected: "Hello from PostgreSQL!" with timestamp.

**Data flow:** Browser → Host:30080 → Kind NodePort → Ingress Controller → path routing
- `/` → frontend-service → Frontend Pod (HTML + JS)
- `/api` → backend-service → Backend Pod → PostgreSQL

## 9. Helm, Prometheus & Grafana

### What is Helm?

Helm is the **package manager for Kubernetes**. Instead of writing 50+ YAML files for Prometheus + Grafana + Alertmanager + ServiceMonitors + RBAC, we use:

```
helm install monitoring prometheus-community/kube-prometheus-stack --namespace monitoring
```

This single command installs:
- **Prometheus** — Pull-based metrics collection and time-series storage
- **Grafana** — Dashboards and visualization
- **Alertmanager** — Alert routing
- **kube-state-metrics** — Cluster state metrics (deployments, pods, etc.)
- **node-exporter** — Node-level hardware/OS metrics
- **Prometheus Operator** — Manages Prometheus instances via custom resources

### Install Helm

On Windows:
```powershell
winget install helm.helm
```
**Restart your terminal** after installation so the PATH is refreshed.

Verify:
```powershell
helm version
# version.BuildInfo{Version:"v3.x.x", ...}
```

### Add Repo & Install kube-prometheus-stack

```powershell
helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
helm repo update
helm install monitoring prometheus-community/kube-prometheus-stack --namespace monitoring
```

Wait for pods to be ready:
```powershell
kubectl get pods -n monitoring -w
# (Ctrl+C when all Running)
```

Expected output:
```
alertmanager-monitoring-kube-prometheus-alertmanager-0    2/2  Running
monitoring-grafana-xxxxxxxxx-xxxxx                       3/3  Running
monitoring-kube-prometheus-operator-xxxxxxxxx-xxxxx      1/1  Running
monitoring-kube-state-metrics-xxxxxxxxx-xxxxx            1/1  Running
monitoring-prometheus-node-exporter-xxxxx                1/1  Running
prometheus-monitoring-kube-prometheus-prometheus-0       2/2  Running
```

### Access Grafana

```powershell
kubectl port-forward -n monitoring svc/monitoring-grafana 3000:80
```

Open http://localhost:3000

**Login:** `admin`

**Password (PowerShell):**
```powershell
[System.Text.Encoding]::UTF8.GetString([System.Convert]::FromBase64String((kubectl get secret -n monitoring monitoring-grafana -o jsonpath="{.data.admin-password}")))
```

**Password (Linux/Mac):**
```bash
kubectl get secret -n monitoring monitoring-grafana -o jsonpath="{.data.admin-password}" | base64 --decode
```

### Explore Dashboards

Go to **Dashboards → Browse** and select:

| Dashboard | What It Shows |
|-----------|---------------|
| **Kubernetes / Compute Resources / Pod** | CPU, memory, network per pod |
| **Kubernetes / Compute Resources / Namespace** | Resource usage grouped by namespace |
| **Kubernetes / Networking / Pod** | Network I/O per pod |
| **Node Exporter / Nodes** | Node-level CPU, memory, disk |

> **Tip:** If graphs show "No data", check the namespace dropdown — make sure `production` is selected, not `default`. Generate traffic by hitting the app if needed.

## 10. Horizontal Pod Autoscaler (HPA)

### Install Metrics Server

HPA needs per-pod metrics. The Metrics Server collects these from kubelets.

```powershell
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

# Fix for kind self-signed kubelet certs:
kubectl patch deployment metrics-server -n kube-system --type='json' -p='[{"op": "add", "path": "/spec/template/spec/containers/0/args/-", "value": "--kubelet-insecure-tls"}]'

# Wait for it:
kubectl -n kube-system wait --for=condition=ready pod -l k8s-app=metrics-server --timeout=60s

# Verify:
kubectl top pods -n production
```

### Create HPA

```powershell
kubectl autoscale deployment backend --cpu-percent=50 --min=2 --max=5 -n production

kubectl get hpa -n production
# NAME      TARGETS   MINPODS   MAXPODS   REPLICAS
# backend   0%/50%    2         5         2
```

**HPA Formula:** `desiredReplicas = ceil[currentReplicas × (currentMetric / targetMetric)]`

Example: 2 Pods at 85% CPU with 50% target:
```
desiredReplicas = ceil[2 × (85 / 50)] = ceil[3.4] = 4
```

### Test with Load Generator

```powershell
kubectl run load-generator --image=busybox -it --rm --restart=Never -n production -- sh -c "while true; do wget -q -O- http://backend-service; done"
```

In another terminal:
```powershell
kubectl get hpa -n production -w
# backend   85%/50%   2
# backend   85%/50%   4    ← scaled up!
# backend   55%/50%   4
```

Stop the load generator to trigger scale-down (after ~5 min cooldown).

## Production Best Practices — Summary

| Practice | Implemented In | Why |
|----------|---------------|-----|
| **Resource requests** | Backend, Frontend, Postgres | Guarantees minimum resources for scheduling |
| **Resource limits** | Backend, Frontend, Postgres | Prevents resource starvation (noisy neighbor) |
| **Liveness probe** | Backend, Frontend, Postgres | Kubelet restarts unhealthy containers |
| **Readiness probe** | Backend, Frontend, Postgres | Service stops sending traffic to unready pods |
| **RBAC** | Backend | Least-privilege ServiceAccount — read-only access to configmaps/secrets/pods |
| **Namespace isolation** | production, monitoring | Logical separation of app and monitoring |
| **Helm** | Prometheus/Grafana install | Reproducible, parameterized deployments |
| **Monitoring** | Prometheus + Grafana | Metrics collection and visualization |
| **HPA** | Backend | Auto-scales under CPU load |
| **Rolling updates** | Default strategy (25% max surge/unavailable) | Zero-downtime deployments |
| **Persistent storage** | Postgres PVC (1Gi) | Data survives pod restarts and rescheduling |

## Complete File Tree

```
10.production-readiness/
├── note.md                              ← Detailed walkthrough
├── note.ipynb                           ← This notebook
├── kind-config.yaml                     ← Kind cluster with port mappings (30080, 30443)
├── k8s/
│   ├── config-and-secret.yaml           ← ConfigMap (DB config) + Secret (password)
│   ├── postgres.yaml                    ← Headless Service + StatefulSet (postgres:15-alpine, 1Gi PVC, probes, limits)
│   ├── backend.yaml                     ← ServiceAccount + Role + RoleBinding + Deployment ×2 + ClusterIP Service
│   ├── frontend.yaml                    ← Deployment ×1 + ClusterIP Service (probes, limits)
│   └── ingress.yaml                     ← NGINX Ingress (/ → frontend, /api → backend)
├── backend/
│   ├── main.py                          ← FastAPI with psycopg2 + PostgreSQL (lifespan pattern)
│   ├── requirements.txt                 ← fastapi, uvicorn, psycopg2-binary
│   └── Dockerfile                       ← python:3.11-alpine, exposes 80
└── frontend/
    ├── main.py                          ← FastAPI with httpx, serves HTML + proxies /api
    ├── requirements.txt                 ← fastapi, uvicorn, httpx
    └── Dockerfile                       ← python:3.11-alpine, exposes 80
```

## Mistakes & Fixes Log

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Helm `get-helm-3` script is bash, not PowerShell | Parse errors in `get_helm.ps1` | Use `winget install helm.helm` on Windows |
| `helm` not recognized after installation | `helm : The term 'helm' is not recognized` | Restart terminal — PATH wasn't refreshed |
| `base64` is a Linux command, not PowerShell | `base64 : The term 'base64' is not recognized` | Use `[System.Text.Encoding]::UTF8.GetString([System.Convert]::FromBase64String(...))` |
| Metrics Server not installed | HPA shows `cpu: <unknown>/50%` | `kubectl apply -f components.yaml` |
| Metrics Server can't connect (kind self-signed certs) | Still `<unknown>` after install | Patch with `--kubelet-insecure-tls` |
| Grafana shows "No data" on dashboards | Empty graphs | Check namespace dropdown is `production` not `default`; generate traffic |
| Prometheus data source missing in Grafana | Dashboards show error | Verify URL is `http://monitoring-kube-prometheus-prometheus.monitoring.svc:9090` |
| Images not loaded into kind | `ErrImagePull` / `ImagePullBackOff` | `kind load docker-image <tag> --name k8-lab` |
| Wrong namespace on Ingress resource | 404 on `localhost:30080` | Ensure Ingress is in `production` namespace |
| Frontend calls wrong backend URL | JS shows error | Frontend uses `http://backend-service:80` (Service DNS name) |
| `kubectl` context wrong | Pods not found in `production` | `kubectl config use-context kind-k8-lab` |

## Key Commands

| Command | Purpose |
|---------|---------|
| `kind delete cluster --name k8-lab && kind create cluster --name k8-lab --config kind-config.yaml` | Recreate cluster with port mappings |
| `kubectl apply -f k8s/` | Apply all manifests (order: config → postgres → backend → frontend → ingress) |
| `kubectl wait --for=condition=ready pod postgres-0 --timeout=60s` | Wait for PostgreSQL to be ready |
| `kubectl -n ingress-nginx wait --for=condition=ready pod --selector=app.kubernetes.io/component=controller --timeout=120s` | Wait for Ingress controller |
| `kubectl -n ingress-nginx patch svc ingress-nginx-controller -p '...'` | Set NodePort to 30080 |
| `docker build -t fastapi-backend:2.0 .` | Build backend image |
| `kind load docker-image fastapi-backend:2.0 --name k8-lab` | Load backend into kind |
| `winget install helm.helm` | Install Helm on Windows |
| `helm install monitoring prometheus-community/kube-prometheus-stack --namespace monitoring` | Install monitoring stack |
| `kubectl port-forward -n monitoring svc/monitoring-grafana 3000:80` | Access Grafana locally |
| `kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml` | Install Metrics Server |
| `kubectl patch deployment metrics-server -n kube-system --type='json' -p='[{"op": "add", "path": "/spec/template/spec/containers/0/args/-", "value": "--kubelet-insecure-tls"}]'` | Fix Metrics Server for kind |
| `kubectl autoscale deployment backend --cpu-percent=50 --min=2 --max=5 -n production` | Create HPA for backend |
| `kubectl get hpa -n production -w` | Watch HPA scaling live |
| `kubectl top pods -n production` | Real-time CPU/memory per pod |

## Key Concepts

| Concept | Key Insight |
|---------|-------------|
| **Liveness Probe** | Tells K8s when to restart a container. Without it, a deadlocked process runs forever. |
| **Readiness Probe** | Tells K8s when a pod can serve traffic. Without it, a pod still starting gets requests and returns errors. |
| **Resource Requests** | Minimum resources guaranteed to a container. Required for HPA and QoS class assignment. |
| **Resource Limits** | Hard cap — container is throttled (CPU) or killed (memory) if it exceeds them. |
| **RBAC ServiceAccount** | Identity for a pod (not a person). Every pod runs under a SA. |
| **RBAC Role** | Namespace-scoped permissions. Grants only what's needed (least privilege). |
| **RBAC RoleBinding** | Binds a Role to subjects (ServiceAccounts, users, groups). |
| **Helm** | Package manager for K8s. One command installs a complex stack (Prometheus + Grafana + Alertmanager + ...). |
| **Prometheus** | Pull-based time-series metrics system. Scrapes targets, stores data, runs queries via PromQL. |
| **Grafana** | Visualization layer. Queries Prometheus and renders dashboards. Pre-built dashboards for K8s. |
| **kube-prometheus-stack** | Helm chart that bundles Prometheus Operator, Prometheus, Grafana, Alertmanager, node-exporter, kube-state-metrics. |
| **Metrics Server** | Required for HPA. Aggregates per-pod CPU/memory from kubelets. Needs `--kubelet-insecure-tls` in kind. |
| **HPA** | Reads metrics from Metrics Server, computes desired replicas, updates Deployment. Scale-up instant; scale-down 5-min cooldown. |
| **Headless Service** | `clusterIP: None`. For StatefulSets — each pod gets a stable DNS name. |
| **StatefulSet** | Stable identity, ordered creation, per-pod PVC via `volumeClaimTemplates`. |
| **ConfigMap + Secret** | External configuration injected as env vars. Keeps config out of the image. |
| **Ingress + NodePort** | Single entry point (host port 30080 → node port 30080 → container port 80). No port-forward needed. |
| **Namespace isolation** | `production` for app, `monitoring` for observability — prevents config conflicts, improves security. |

## Next Up — Final Capstone

The Final Capstone ties everything together:
- Deploy the full stack on **AWS EKS** (managed Kubernetes)
- Implement a **CI/CD pipeline** with GitHub Actions (build → push → deploy on push)
- Add **canary deployments** with traffic splitting
- Document the entire system architecture

> Ready to move on? Let me know!